# E0 — Каноническая постановка и метрики (EPI)

Переход от ошибки уставки к **экономическому показателю EPI** (`EUR/m2·сезон`), считываемому из экономики симулятора `gl_gym` (`GreenhouseReward`): `EPI = sum(profit)` за сезон, с декомпозицией выручки и затрат на тепло/CO2/электричество и метриками коридоров T/CO2/RH. Здесь же замораживается конфиг протокола (`protocol.json`).

In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P

# FAST_MODE smoke (tiny data) vs article-grade. Toggle via env var ARTICLE_FAST=0.
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location)
CORR, PRICES = ECON["corridors"], ECON["prices"]
print("FAST_MODE", FAST_MODE, "| seeds", tuple(pc.seeds), "| budgets", pc.budgets_days,
      "| n_days_train/test", pc.n_days_train, pc.n_days_test)

FAST_MODE False | seeds (0, 1, 2, 3, 4, 5, 6, 7, 8, 9) | budgets (1, 3, 7, 14, 30, 60) | n_days_train/test 60 60


## Конфиг протокола: цены и коридоры из живого симулятора

In [2]:
protocol = {"config": pc.to_dict(), "economics": ECON,
            "primary_metric": "EPI = sum(profit) [EUR/m2.season] from gl_gym GreenhouseReward",
            "split": {"train_years": list(pc.train_years), "test_year": pc.test_year,
                       "ood_years": list(pc.ood_years), "note": "leakage-free year split"}}
U.save_json(RES / "protocol.json", protocol)
print(json.dumps(ECON, indent=2, ensure_ascii=False))

{
  "location": "Rostov-on-Don",
  "corridors": {
    "co2": [
      300.0,
      1600.0
    ],
    "t_in": [
      15.0,
      34.0
    ],
    "rh": [
      50.0,
      85.0
    ]
  },
  "prices": {
    "fruit_price_eur_per_kg": 1.6,
    "heating_price_eur_per_kwh": 0.09,
    "elec_price_eur_per_kwh": 0.3,
    "co2_price_eur_per_kg": 0.3,
    "dmfm": 0.065
  }
}


## Демонстрация EPI на эталонном rule-based сезоне (тест-год, in-distribution)

In [3]:
scen = pc.test_scenario()
cfg = pc.cfg_for(scen, seed=0)
data = U.collect_rule_based_dataset(cfg, n_days=pc.n_days_test, prbs_scale=0.0)
df = data.to_frame()
m = U.epi_metrics(df, corridors=CORR, prices=PRICES)
epi_tbl = pd.DataFrame([m])
U.save_table(epi_tbl, RES / "tables" / "e0_epi_decomposition.csv")
cols = ["epi","revenue","cost_total","cost_heat","cost_co2","cost_elec",
        "energy_heat_kwh_m2","energy_elec_kwh_m2","co2_kg_m2","fruit_dm_growth",
        "t_in_in_corridor_pct","co2_in_corridor_pct","rh_in_corridor_pct","violation_steps_total"]
epi_tbl[cols].T

,0
epi,1.640412
revenue,18.882339
cost_total,17.241927
cost_heat,8.412143
cost_co2,2.580147
cost_elec,6.249637
energy_heat_kwh_m2,93.468261
energy_elec_kwh_m2,20.832122
co2_kg_m2,8.600491
fruit_dm_growth,767095.040700


## Рисунок: накопленный EPI и декомпозиция затрат

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df["time_h"] / 24.0, df["profit"].cumsum())
axes[0].set_title("Накопленный EPI"); axes[0].set_xlabel("сутки"); axes[0].set_ylabel("EUR/m2"); axes[0].grid(alpha=.3)
axes[1].bar(["heat","elec","co2"], [m["cost_heat"], m["cost_elec"], m["cost_co2"]])
axes[1].set_title("Затраты, EUR/m2")
U.save_figure(fig, RES / "figures" / "e0_epi_breakdown.png"); plt.close(fig)
print("EPI(rule-based) =", round(m["epi"], 4), "EUR/m2 |  saved e0_epi_breakdown.png")

EPI(rule-based) = 1.6404 EUR/m2 |  saved e0_epi_breakdown.png


**Итог E0.** Первичная метрика EPI и её декомпозиция считаются из экономики симулятора; коридоры T/CO2/RH — из `env.constraints_*`. Конфиг заморожен в `protocol.json`.